# 03 · Metrica-Beispielspiele: Labels aus Events

Die drei öffentlichen Metrica-Sports-Beispielspiele haben echte Ausfälle im Tracking und liegen in Unterbrechungen näher an der Challenge als das degradierte DFL. Sie werden deshalb **ohne Degradation** als zusätzliche Trainingsspiele genutzt. Frame-Labels gibt es dort nicht; wir leiten sie automatisch aus den Metrica-Events ab.

**Ausgabe:** `artifacts/metrica/MET{1,2,3}.parquet`, Trainingsdaten für Notebook 04.

In [1]:
# Standard library
import json
from pathlib import Path

# Data
import numpy as np
import pandas as pd

# Tracking loader
from kloppy import metrica

ROOT = Path.cwd().resolve().parent
MET_RAW = ROOT / "external_data" / "metrica_sample" / "raw"
MET_OUT = ROOT / "artifacts" / "metrica"
MET_OUT.mkdir(parents=True, exist_ok=True)

FPS = 25
PITCH_MET = np.array([105.0, 68.0])  # Metadaten der Beispielspiele
MAX_BALL_SPEED, MAX_PLAYER_SPEED = 40.0, 12.0  # wie Notebook 02
POSS_TYPES = {"PASS", "CARRY", "RECOVERY", "SET PIECE", "SHOT", "BALL LOST", "BALL OUT"}
STOP_LOST = {"OFFSIDE", "HAND BALL", "END HALF"}

## 1. Tracking und Events einlesen

Spiel 1 und 2 liegen als CSV vor, Spiel 3 im EPTS-Format. Koordinaten wie bei der Challenge: `(v − 0,5) · Feldgröße`.

In [2]:
def clean(ball, players):
    ball, players = ball.copy(), players.copy()
    ball[np.r_[0, np.linalg.norm(np.diff(ball, axis=0), axis=1) * FPS] > MAX_BALL_SPEED] = np.nan
    step = np.linalg.norm(np.diff(players, axis=0), axis=-1) * FPS
    players[np.r_[np.zeros((1, players.shape[1])), step] > MAX_PLAYER_SPEED] = np.nan
    return ball, players

def tracking(g):
    if g == 3:
        ds = metrica.load_tracking_epts(meta_data=MET_RAW / "Sample_Game_3_metadata.xml", raw_data=MET_RAW / "Sample_Game_3_tracking.txt")
    else:
        ds = metrica.load_tracking_csv(home_data=MET_RAW / f"Sample_Game_{g}_RawTrackingData_Home_Team.csv",
                                       away_data=MET_RAW / f"Sample_Game_{g}_RawTrackingData_Away_Team.csv")
    df = ds.to_df(engine="pandas")
    ids = [(s, p.player_id) for s, t in zip("HA", ds.metadata.teams) for p in t.players if f"{p.player_id}_x" in df]
    xy = lambda cols: (np.stack([df[[f"{c}_{a}" for c in cols]].to_numpy(float) for a in "xy"], -1) - 0.5) * PITCH_MET
    return df["frame_id"].to_numpy(), df["period_id"].to_numpy(), xy(["ball"])[:, 0], xy([i for _, i in ids]), np.array([s for s, _ in ids]), ds.metadata.teams[0].name

def events(g, home):
    if g == 3:
        sub = lambda s: "" if not s else s["name"] if isinstance(s, dict) else "-".join(x["name"] for x in s)
        e = pd.DataFrame([{"team": "H" if x["team"]["name"] == home else "A", "type": x["type"]["name"], "sub": sub(x["subtypes"]),
                           "period": x["period"], "start": x["start"]["frame"], "end": x["end"]["frame"]}
                          for x in json.load(open(MET_RAW / "Sample_Game_3_events.json"))["data"]])
    else:
        r = pd.read_csv(MET_RAW / f"Sample_Game_{g}_RawEventsData.csv")
        e = pd.DataFrame({"team": r["Team"].map({"Home": "H", "Away": "A"}), "type": r["Type"], "sub": r["Subtype"].fillna(""),
                          "period": r["Period"], "start": r["Start Frame"], "end": r["End Frame"]})
    e["end"] = np.maximum(e["start"], e["end"])  # Anstöße haben in Spiel 1/2 End Frame 0
    return e.sort_values("start", kind="stable").reset_index(drop=True)

raw = {}
for g in (1, 2, 3):
    frame, period, ball, players, side, home = tracking(g)
    raw[g] = {"frame": frame, "period": period, "ball": ball, "players": players, "side": side, "events": events(g, home)}
    print(f"Spiel {g}: {len(frame):,} Frames · {len(side)} Spieler · {len(raw[g]['events']):,} Events · Ball fehlt in {np.isnan(ball[:, 0]).mean():.1%}")

Spiel 1: 145,006 Frames · 28 Spieler · 1,745 Events · Ball fehlt in 39.1%


Spiel 2: 141,156 Frames · 26 Spieler · 1,935 Events · Ball fehlt in 41.0%


Spiel 3: 143,761 Frames · 35 Spieler · 3,620 Events · Ball fehlt in 32.2%


## 2. Labelregel

- **`D` beginnt** am Ende von `BALL OUT`, bei einem Schuss ins Aus oder Tor, bei einem Foul (`CHALLENGE …FAULT…`, `FAULT RECEIVED`) und bei Abseits, Handspiel oder Halbzeitende.
- **`D` endet** beim nächsten `SET PIECE` oder Abstoß.
- **Im Spiel** gehört der Ball dem Team des letzten Besitzereignisses (Pass, Dribbling, Ballgewinn, Standard, Schuss, Ballverlust). Zweikämpfe, Karten und Fouls übertragen keinen Besitz.

Bewertet wird je Halbzeit vom ersten Standard (Anstoß) bis zum letzten Event. Danach dieselbe Plausibilitätsregel wie in Notebook 02.

In [3]:
def points(e):
    tok = e["sub"].str.split("-").apply(set)
    stop_end = (e["type"] == "BALL OUT") | ((e["type"] == "SHOT") & tok.apply(lambda t: bool(t & {"OUT", "GOAL"}))) | \
               ((e["type"] == "BALL LOST") & tok.apply(lambda t: bool(t & STOP_LOST)))
    stop_start = ((e["type"] == "CHALLENGE") & tok.apply(lambda t: "FAULT" in t)) | (e["type"] == "FAULT RECEIVED")
    restart = (e["type"] == "SET PIECE") | (e["type"].isin(["PASS", "BALL LOST", "BALL OUT"]) & tok.apply(lambda t: "GOAL KICK" in t))
    p = pd.concat([pd.DataFrame({"frame": e.loc[stop_end, "end"], "kind": 0}), pd.DataFrame({"frame": e.loc[stop_start, "start"], "kind": 0}),
                   pd.DataFrame({"frame": e.loc[restart, "start"], "kind": 1})])
    return p.sort_values(["frame", "kind"]).to_numpy()

def label_game(r):
    e, frame = r["events"], r["frame"]
    keep, lab = np.zeros(len(frame), bool), np.full(len(frame), "", dtype="<U1")
    for per in (1, 2):
        ep = e[e["period"] == per]
        first = ep.loc[(ep["type"] == "SET PIECE"), "start"].min()
        inside = (r["period"] == per) & (frame >= first) & (frame <= ep["end"].max())
        keep |= inside

        poss = ep[ep["type"].isin(POSS_TYPES)]
        j = np.searchsorted(poss["start"].to_numpy(), frame[inside], side="right") - 1
        lab[inside] = poss["team"].to_numpy()[np.maximum(j, 0)]

        dead, open_at = np.zeros(len(frame), bool), None
        for f, kind in points(ep):
            if kind == 0 and open_at is None:
                open_at = f
            elif kind == 1 and open_at is not None:
                dead |= (frame >= open_at) & (frame < f)
                open_at = None
        if open_at is not None:
            dead |= frame >= open_at
        lab[inside & dead] = "D"
    return keep, lab

summary = {}
for g, r in raw.items():
    keep, lab = label_game(r)
    ball, players = r["ball"][keep], r["players"][keep]
    per = r["period"][keep]
    cb, cp = zip(*[clean(ball[per == p], players[per == p]) for p in (1, 2)])
    ball, players = np.concatenate(cb), np.concatenate(cp)

    cols = {"frame": r["frame"][keep], "period": per, "label": lab[keep], "ball_x": ball[:, 0], "ball_y": ball[:, 1]}
    for k, s in enumerate(r["side"]):
        cols[f"{s}_P{k:02d}_x"], cols[f"{s}_P{k:02d}_y"] = players[:, k, 0], players[:, k, 1]
    pd.DataFrame(cols).to_parquet(MET_OUT / f"MET{g}.parquet", index=False)
    summary[f"MET{g}"] = {"Minuten": keep.sum() / FPS / 60, **pd.Series(lab[keep]).value_counts(normalize=True).to_dict()}

pd.DataFrame(summary).T.round(3)

,Minuten,D,H,A
MET1,94.716,0.449,0.278,0.273
MET2,93.409,0.415,0.303,0.282
MET3,95.132,0.298,0.366,0.337


**Ergebnis:** Drei Spiele mit automatisch abgeleiteten H/A/D-Labels. Die Labelregel stammt aus den Metrica-Events, nicht aus der Challenge. Die Event-Zeitstempel sind menschlich erfasst und nicht framegenau.